In [0]:
# =============================================================
# schema_manager.py
# Layer     : Bronze
# Purpose   : Schema governance only.
#             - Owns StructType definitions for all 14 tables
#             - Provides get_schema(table_name)
#             - Handles schema evolution via compare_schema()
#
# STRICT RULE: No spark.read, no spark.write, no Delta here.
#              Only StructType / StructField definitions.
# =============================================================

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    DoubleType, BooleanType, TimestampType, DateType
)

# ── Schema registry ──────────────────────────────────────────────────────────
# Each schema reflects the exact columns written by the Python generators.
# Anomaly injection may produce nulls or bad values in fact tables —
# that is intentional; Silver layer fixes them.

_SCHEMAS = {

    # ── DIMENSION TABLES ────────────────────────────────────────────────────

    "owners": StructType([
        StructField("owner_id",           StringType(),    False),
        StructField("owner_name",         StringType(),    True),
        StructField("email",              StringType(),    True),
        StructField("phone",              StringType(),    True),
        StructField("city",               StringType(),    True),
        StructField("state",              StringType(),    True),
        StructField("country",            StringType(),    True),
        StructField("registration_date",  StringType(),    True),
    ]),

    "pets": StructType([
        StructField("pet_id",      StringType(),  False),
        StructField("owner_id",    StringType(),  True),
        StructField("pet_name",    StringType(),  True),
        StructField("species",     StringType(),  True),
        StructField("breed",       StringType(),  True),
        StructField("gender",      StringType(),  True),
        StructField("birth_date",  StringType(),  True),
        StructField("weight_kg",   StringType(),  True),   # kept String — anomalies may inject text
    ]),

    "products": StructType([
        StructField("product_id",           StringType(),  False),
        StructField("product_name",         StringType(),  True),
        StructField("device_category",      StringType(),  True),
        StructField("manufacturing_cost",   StringType(),  True),
        StructField("msrp",                 StringType(),  True),
    ]),

    "firmware": StructType([
        StructField("firmware_id",      StringType(),  False),
        StructField("firmware_version", StringType(),  True),
        StructField("release_date",     StringType(),  True),
        StructField("release_type",     StringType(),  True),
    ]),

    "batches": StructType([
        StructField("batch_id",          StringType(),  False),
        StructField("factory_name",      StringType(),  True),
        StructField("factory_location",  StringType(),  True),
        StructField("production_date",   StringType(),  True),
    ]),

    "devices": StructType([
        StructField("device_id",          StringType(),  False),
        StructField("pet_id",             StringType(),  True),
        StructField("product_id",         StringType(),  True),
        StructField("firmware_id",        StringType(),  True),
        StructField("batch_id",           StringType(),  True),
        StructField("purchase_date",      StringType(),  True),
        StructField("activation_date",    StringType(),  True),
        StructField("warranty_end_date",  StringType(),  True),
    ]),

    # ── FACT TABLES ─────────────────────────────────────────────────────────
    # All numeric columns kept as StringType at Bronze.
    # Silver casts to proper types after anomaly cleaning.

    "sales": StructType([
        StructField("sale_id",          StringType(),  False),
        StructField("sale_date",        StringType(),  True),
        StructField("device_id",        StringType(),  True),
        StructField("owner_id",         StringType(),  True),
        StructField("sale_price",       StringType(),  True),
        StructField("discount_amount",  StringType(),  True),
        StructField("sales_channel",    StringType(),  True),
        StructField("city",             StringType(),  True),
        StructField("state",            StringType(),  True),
    ]),

    "device_snapshots": StructType([
        StructField("snapshot_id",         StringType(),  False),
        StructField("snapshot_timestamp",  StringType(),  True),
        StructField("device_id",           StringType(),  True),
        StructField("battery_pct",         StringType(),  True),
        StructField("signal_strength",     StringType(),  True),
        StructField("temperature_c",       StringType(),  True),
    ]),

    "activity_snapshots": StructType([
        StructField("activity_snapshot_id",  StringType(),  False),
        StructField("snapshot_timestamp",    StringType(),  True),
        StructField("device_id",             StringType(),  True),
        StructField("pet_id",                StringType(),  True),
        StructField("steps",                 StringType(),  True),
        StructField("distance_km",           StringType(),  True),
        StructField("active_minutes",        StringType(),  True),
    ]),

    "health_snapshots": StructType([
        StructField("health_snapshot_id",  StringType(),  False),
        StructField("snapshot_timestamp",  StringType(),  True),
        StructField("device_id",           StringType(),  True),
        StructField("pet_id",              StringType(),  True),
        StructField("heart_rate",          StringType(),  True),
        StructField("pulse_rate",          StringType(),  True),
        StructField("body_temperature",    StringType(),  True),
    ]),

    "feeding_events": StructType([
        StructField("feed_event_id",          StringType(),  False),
        StructField("event_timestamp",        StringType(),  True),
        StructField("device_id",              StringType(),  True),
        StructField("pet_id",                 StringType(),  True),
        StructField("food_dispensed_grams",   StringType(),  True),
    ]),

    "hydration_events": StructType([
        StructField("water_event_id",      StringType(),  False),
        StructField("event_timestamp",     StringType(),  True),
        StructField("device_id",           StringType(),  True),
        StructField("pet_id",              StringType(),  True),
        StructField("water_consumed_ml",   StringType(),  True),
    ]),

    "device_failures": StructType([
        StructField("failure_id",          StringType(),  False),
        StructField("device_id",           StringType(),  True),
        StructField("failure_timestamp",   StringType(),  True),
        StructField("failure_type",        StringType(),  True),
        StructField("downtime_minutes",    StringType(),  True),
        StructField("severity_score",      StringType(),  True),
    ]),

    "firmware_deployments": StructType([
        StructField("deployment_id",                  StringType(),  False),
        StructField("device_id",                      StringType(),  True),
        StructField("firmware_id",                    StringType(),  True),
        StructField("deployment_timestamp",           StringType(),  True),
        StructField("deployment_status",              StringType(),  True),
        StructField("rollback_flag",                  StringType(),  True),
        StructField("deployment_duration_seconds",    StringType(),  True),
    ]),
}

# ── Public API ────────────────────────────────────────────────────────────────

def get_schema(table_name: str) -> StructType:
    """
    Return the StructType schema for a given table.
    Raises KeyError with a helpful message if table is unknown.
    """
    if table_name not in _SCHEMAS:
        available = sorted(_SCHEMAS.keys())
        raise KeyError(
            f"[schema_manager] Unknown table: '{table_name}'. "
            f"Available: {available}"
        )
    return _SCHEMAS[table_name]


def list_tables() -> list:
    """Return all registered table names."""
    return sorted(_SCHEMAS.keys())


def compare_schema(table_name: str, incoming_df) -> dict:
    """
    Compare the registered schema against the schema of an incoming DataFrame.
    Returns a dict with:
        - missing_cols  : columns in registry but not in df
        - extra_cols    : columns in df but not in registry
        - type_mismatches: columns present in both but with different types
    Useful for detecting schema drift before writing to Bronze.
    """
    registered = {f.name: f.dataType for f in get_schema(table_name).fields}
    incoming   = {f.name: f.dataType for f in incoming_df.schema.fields
                  if not f.name.startswith("_")}   # ignore lineage cols

    missing_cols    = [c for c in registered if c not in incoming]
    extra_cols      = [c for c in incoming   if c not in registered]
    type_mismatches = [
        {"column": c, "expected": str(registered[c]), "found": str(incoming[c])}
        for c in registered
        if c in incoming and registered[c] != incoming[c]
    ]

    return {
        "table":           table_name,
        "missing_cols":    missing_cols,
        "extra_cols":      extra_cols,
        "type_mismatches": type_mismatches,
        "is_clean":        not (missing_cols or extra_cols or type_mismatches),
    }


def validate_schema(table_name: str, incoming_df) -> None:
    """
    Calls compare_schema and raises RuntimeError if critical drift found.
    Extra columns are allowed (schema evolution).
    Missing columns or type mismatches raise an error.
    """
    result = compare_schema(table_name, incoming_df)

    if result["missing_cols"]:
        raise RuntimeError(
            f"[schema_manager] Schema drift detected for '{table_name}'. "
            f"Missing columns: {result['missing_cols']}"
        )
    if result["type_mismatches"]:
        raise RuntimeError(
            f"[schema_manager] Type mismatch for '{table_name}': "
            f"{result['type_mismatches']}"
        )
    if result["extra_cols"]:
        print(f"[schema_manager] INFO: New columns detected in '{table_name}': "
              f"{result['extra_cols']} — allowed via schema evolution.")

print("[schema_manager] Loaded. Tables registered:", list_tables())